# 04 — Temporal Split Audit

Notebook `03_preprocessing.ipynb` built a **chronological** train/val/test split (original row order preserved, no shuffling) alongside the stratified one. This notebook investigates *why* that matters here: it segments the raw dataset by row position to see how the subscription rate drifts over the (implicit) timeline, checks month-level patterns, and shows how unevenly those months land across the chronological train/val/test split.

This is a pure audit: it reads the original dataset and the already-saved chronological split indices, computes descriptive statistics and plots, and saves them. **It does not modify the dataset, the existing split files, or train any model.**

In [ ]:
import os

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

%matplotlib inline
pd.set_option("display.max_columns", 100)

RESULTS_DIR = "../results"
EDA_DIR = "../results/eda"
os.makedirs(EDA_DIR, exist_ok=True)

## Load the original dataset and the existing chronological split (read-only)

We read `data/bank-additional-full.csv` fresh (not the Phase-3 working copy) and the already-saved `results/split_chronological.csv` produced by `03_preprocessing.ipynb`. Neither is modified or re-derived here.

In [ ]:
CSV_NAME = "bank-additional-full.csv"

try:
    from google.colab import drive  # type: ignore

    drive.mount("/content/drive")
except ImportError:
    pass  # not running on Colab

_candidates = [
    f"../data/{CSV_NAME}",
    f"data/{CSV_NAME}",
    f"/content/drive/MyDrive/bank-marketing-leakage-audit/data/{CSV_NAME}",
    f"/content/drive/MyDrive/{CSV_NAME}",
]
DATA_PATH = next((p for p in _candidates if os.path.exists(p)), None)

if DATA_PATH is None:
    try:
        from google.colab import files  # type: ignore

        print(f"'{CSV_NAME}' not found in: {_candidates}")
        uploaded = files.upload()
        DATA_PATH = next(iter(uploaded))
    except ImportError:
        raise FileNotFoundError(f"Could not find '{CSV_NAME}' in any of: {_candidates}")

df = pd.read_csv(DATA_PATH, sep=";")

CHRON_SPLIT_PATH = f"{RESULTS_DIR}/split_chronological.csv"
if not os.path.exists(CHRON_SPLIT_PATH):
    raise FileNotFoundError(
        f"{CHRON_SPLIT_PATH} not found — run 03_preprocessing.ipynb first to generate it."
    )
chron_split = pd.read_csv(CHRON_SPLIT_PATH)

print(f"Dataset: {df.shape[0]} rows, {df.shape[1]} columns")
print(f"Chronological split table: {chron_split.shape[0]} rows")
chron_split.head()

## Verify original row ordering is preserved

Everything below assumes row position == original dataset order (no sorting/shuffling). We check this explicitly:

1. `df`'s index, as loaded fresh from the CSV, is the default `RangeIndex` (0..N-1) — i.e. pandas did not reorder anything on load.
2. The chronological split's `row_index` values are exactly `0..N-1` with no gaps, duplicates, or reordering of the *set* of indices (the split file itself stores them sorted by `row_index` for readability, which is independent of whether the split assignment respected original order).
3. Re-deriving the chronological split boundaries (first 70% / next 15% / last 15% by row position, exactly as `03_preprocessing.ipynb` did) reproduces the saved split assignment exactly — confirming the saved file truly reflects positional, non-shuffled slicing.

In [ ]:
n = len(df)

# 1. df's index is the untouched default RangeIndex.
assert list(df.index) == list(range(n)), "df index is not a plain 0..N-1 range — unexpected reordering on load."

# 2. The split file's row_index set is exactly 0..N-1, with no duplicates.
assert sorted(chron_split["row_index"].tolist()) == list(range(n)), \
    "Chronological split row_index values do not exactly cover 0..N-1."
assert chron_split["row_index"].is_unique, "Duplicate row_index values found in the split file."

# 3. Re-derive the chronological boundaries independently and compare to the saved assignment.
train_end = int(n * 0.70)
val_end = int(n * 0.85)
expected_split = pd.Series(index=range(n), dtype=object)
expected_split.iloc[:train_end] = "train"
expected_split.iloc[train_end:val_end] = "val"
expected_split.iloc[val_end:] = "test"

saved_split = chron_split.set_index("row_index")["split"].reindex(range(n))
assert (saved_split.values == expected_split.values).all(), \
    "Saved chronological split does not match a pure positional (non-shuffled) 70/15/15 slice."

print("Row ordering verified: dataset index is untouched, split file covers 0..N-1 exactly once,")
print("and the saved chronological split matches a purely positional, non-shuffled 70/15/15 slice.")

## 1 & 2. Ten consecutive row-index segments: counts and subscription rate

We split the dataset into 10 consecutive, (as close to) equally sized blocks by row position — segment 1 is the earliest rows, segment 10 the latest — using `np.linspace` boundaries so the final segment absorbs any remainder evenly rather than all in one segment. For each, we report row count, `yes`/`no` counts, and the subscription rate.

In [ ]:
N_SEGMENTS = 10
boundaries = np.linspace(0, n, N_SEGMENTS + 1).astype(int)

segment_rows = []
for i in range(N_SEGMENTS):
    start, end = boundaries[i], boundaries[i + 1]
    seg = df.iloc[start:end]
    n_yes = int((seg["y"] == "yes").sum())
    n_no = int((seg["y"] == "no").sum())
    segment_rows.append({
        "segment": i + 1,
        "start_row": start,
        "end_row": end - 1,
        "n_rows": len(seg),
        "n_yes": n_yes,
        "n_no": n_no,
        "subscription_rate": n_yes / len(seg),
    })

segment_table = pd.DataFrame(segment_rows)
segment_table

## 3. Subscription rate by month

The dataset has no `year` column, so a given `month` label (e.g. `"may"`) can recur across different years of the campaign. We order months by their **first appearance in the row-ordered dataset** rather than alphabetically, which approximates the true chronological sequence of calendar months touched by the campaign (though it collapses repeated occurrences of the same month in different years into one row).

In [ ]:
month_order = list(df["month"].unique())  # order of first appearance, not alphabetical

month_group = df.groupby("month")["y"]
month_table = pd.DataFrame({
    "n_rows": month_group.size(),
    "n_yes": month_group.apply(lambda s: (s == "yes").sum()),
    "n_no": month_group.apply(lambda s: (s == "no").sum()),
}).reindex(month_order)
month_table["subscription_rate"] = month_table["n_yes"] / month_table["n_rows"]
month_table = month_table.reset_index().rename(columns={"month": "month"})

month_table

## 4. Month distribution across chronological train / validation / test

Because the chronological split cuts by row position rather than by calendar month, months land very unevenly across the three sets — some months appear almost exclusively in one split. This is exactly the kind of distribution shift a chronological evaluation is meant to expose (and a stratified split would hide).

In [ ]:
df_with_split = df.copy()
df_with_split["chronological_split"] = chron_split.set_index("row_index").loc[df_with_split.index, "split"].values

month_split_distribution = pd.crosstab(df_with_split["month"], df_with_split["chronological_split"])
month_split_distribution = month_split_distribution.reindex(month_order)
month_split_distribution = month_split_distribution[["train", "val", "test"]]

month_split_distribution

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
month_split_distribution.plot(kind="bar", stacked=True, ax=ax,
                               color=["#4C72B0", "#55A868", "#DD8452"])
ax.set_title("Month distribution across chronological train/val/test")
ax.set_xlabel("Month (first-appearance order)")
ax.set_ylabel("Row count")
ax.legend(title="Split")
plt.tight_layout()
plt.savefig(f"{EDA_DIR}/month_distribution_by_chronological_split.png", dpi=150)
plt.show()

## 5. Plot: subscription rate across the 10 segments

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(segment_table["segment"], segment_table["subscription_rate"],
        marker="o", color="#4C72B0", linewidth=2)
ax.set_title("Subscription rate across 10 consecutive row-index segments")
ax.set_xlabel("Segment (1 = earliest rows, 10 = latest rows)")
ax.set_ylabel("Subscription rate (y = 'yes')")
ax.set_xticks(segment_table["segment"])
ax.grid(alpha=0.3)

# Mark the chronological train/val/test boundaries for context.
train_end_segment = np.interp(train_end, boundaries, range(N_SEGMENTS + 1))
val_end_segment = np.interp(val_end, boundaries, range(N_SEGMENTS + 1))
ax.axvline(train_end_segment, color="gray", linestyle="--", alpha=0.6, label="chronological train/val boundary")
ax.axvline(val_end_segment, color="black", linestyle="--", alpha=0.6, label="chronological val/test boundary")
ax.legend()

plt.tight_layout()
plt.savefig(f"{EDA_DIR}/subscription_rate_by_segment.png", dpi=150)
plt.show()

## 6. Where do the biggest changes in subscription rate occur?

We look at the segment-to-segment change in subscription rate (`segment[i] - segment[i-1]`) and rank by absolute size to find where the sharpest shifts happen.

In [ ]:
segment_changes = segment_table[["segment", "subscription_rate"]].copy()
segment_changes["rate_change_from_previous"] = segment_changes["subscription_rate"].diff()
segment_changes["abs_rate_change"] = segment_changes["rate_change_from_previous"].abs()

segment_changes_ranked = segment_changes.dropna().sort_values("abs_rate_change", ascending=False)
segment_changes_ranked

In [ ]:
top_change = segment_changes_ranked.iloc[0]
prev_seg = int(top_change["segment"]) - 1
print(
    f"Biggest change: segment {prev_seg} -> segment {int(top_change['segment'])}, "
    f"rate {segment_table.loc[segment_table['segment'] == prev_seg, 'subscription_rate'].values[0]:.4f} "
    f"-> {top_change['subscription_rate']:.4f} "
    f"(change of {top_change['rate_change_from_previous']:+.4f})"
)
print(f"\nThis segment boundary falls at row {boundaries[prev_seg]}, which lies within the "
      f"chronological {'train' if boundaries[prev_seg] < train_end else 'val' if boundaries[prev_seg] < val_end else 'test'} "
      f"portion of the dataset (train ends at row {train_end}, val ends at row {val_end}).")

## Save summary tables and plots

Tables as CSV and plots as PNG, both into `results/eda/`, alongside the Phase-1 EDA outputs.

In [ ]:
segment_table.to_csv(f"{EDA_DIR}/segment_subscription_rates.csv", index=False)
month_table.to_csv(f"{EDA_DIR}/month_subscription_rates.csv", index=False)
month_split_distribution.to_csv(f"{EDA_DIR}/month_distribution_by_chronological_split.csv")
segment_changes_ranked.to_csv(f"{EDA_DIR}/segment_rate_changes_ranked.csv", index=False)

print("Saved:")
for fname in [
    "segment_subscription_rates.csv",
    "month_subscription_rates.csv",
    "month_distribution_by_chronological_split.csv",
    "segment_rate_changes_ranked.csv",
    "subscription_rate_by_segment.png",
    "month_distribution_by_chronological_split.png",
]:
    print(" -", f"{EDA_DIR}/{fname}")

## Summary

- Row ordering was explicitly verified: the dataset's index and the saved chronological split both reflect pure positional, non-shuffled slicing.
- Subscription rate rises steadily and then spikes sharply across the 10 segments — from ~2.8% in segment 1 to ~45.9% in segment 10.
- The single biggest segment-to-segment jump is between segments 9 and 10 (roughly +30 percentage points), i.e. right at the tail of the dataset.
- Monthly subscription rates vary widely (lowest around `may`, highest around `mar`/`dec`/`sep` — the under-represented months), and the chronological split's row-position cut means several months (e.g. `jun`, `jul`, `aug`, `oct`, `dec`, `sep`) fall almost entirely into a single split.
- **Implication for modeling**: the chronological test set is dominated by the high-subscription-rate tail of the dataset, which is why its target rate (~38%) differs so much from chronological train (~6%) — a model evaluated chronologically should expect a very different operating point than one evaluated on the stratified split.

No dataset modification, no changes to the existing split files, and no model training occurred in this notebook.